# Blackjack Simulator — Analysis Notebook

Interactive analysis of blackjack strategies and card counting systems using Monte Carlo simulation.

---

In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
from blackjack import run_simulation, Strategy, CountingSystem

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print('Imports ready.')

## 1 · Earnings over time — single run

In [ ]:
stats = run_simulation(
    num_hands=10_000,
    base_bet=25,
    num_decks=6,
    strategy=Strategy.BASIC,
    counting_system=CountingSystem.HILOW,
    bet_spread=4,
    track_every=100,
)

fig, ax = plt.subplots(figsize=(12, 4))
x = np.linspace(0, 10_000, len(stats.earnings_history))
color = '#22c55e' if stats.net_earnings >= 0 else '#ef4444'
ax.plot(x, stats.earnings_history, color=color, lw=1.5, label='Net earnings')
ax.axhline(0, color='gray', lw=0.8, linestyle='--')
ax.fill_between(x, stats.earnings_history, 0,
                where=[v >= 0 for v in stats.earnings_history],
                alpha=0.15, color='#22c55e')
ax.fill_between(x, stats.earnings_history, 0,
                where=[v < 0 for v in stats.earnings_history],
                alpha=0.15, color='#ef4444')
ax.set_title(f'Earnings — Basic Strategy + Hi-Lo  |  Net: ${stats.net_earnings:+,.0f}')
ax.set_xlabel('Hands played')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

s = stats.summary()
print(f"Win rate:   {s['win_rate']:.1f}%")
print(f"House edge: {s['house_edge']:.3f}%")
print(f"Blackjacks: {s['blackjacks']:,}")

## 2 · Strategy comparison

In [ ]:
HANDS = 20_000
BET   = 25

strategies = [
    (Strategy.BASIC,      'Basic strategy'),
    (Strategy.ALWAYS17,   'Always stand 17+'),
    (Strategy.MIMIC,      'Mimic dealer'),
    (Strategy.NEVER_BUST, 'Never bust'),
    (Strategy.RANDOM,     'Random'),
]

results = []
for strat, label in strategies:
    s = run_simulation(num_hands=HANDS, base_bet=BET, strategy=strat)
    results.append((label, s.net_earnings, s.win_rate, s.house_edge))
    print(f'{label:<22} net={s.net_earnings:>+8,.0f}  edge={s.house_edge:.2f}%')

labels  = [r[0] for r in results]
edges   = [r[3] for r in results]
colors  = ['#22c55e' if e <= 0 else '#ef4444' for e in edges]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(labels, edges, color=colors, width=0.5, edgecolor='none')
ax.axhline(0, color='gray', lw=0.8)
ax.set_ylabel('House edge (%)')
ax.set_title(f'House edge by strategy ({HANDS:,} hands, ${BET} bet)')
for bar, val in zip(bars, edges):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val:.2f}%', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

## 3 · Card counting systems comparison

In [ ]:
systems = [
    (CountingSystem.NONE,   1,  'No counting'),
    (CountingSystem.HILOW,  4,  'Hi-Lo  (1-4)'),
    (CountingSystem.KO,     4,  'KO     (1-4)'),
    (CountingSystem.HIOPT1, 4,  'Hi-Opt I (1-4)'),
    (CountingSystem.OMEGA2, 6,  'Omega II (1-6)'),
    (CountingSystem.ZEN,    6,  'Zen Count (1-6)'),
]

HANDS = 30_000
count_results = []
for sys_enum, spread, label in systems:
    s = run_simulation(
        num_hands=HANDS, base_bet=25,
        strategy=Strategy.BASIC,
        counting_system=sys_enum,
        bet_spread=spread,
        track_every=500,
    )
    count_results.append((label, s))
    print(f'{label:<22} net={s.net_earnings:>+9,.0f}  edge={s.house_edge:.3f}%')

fig, ax = plt.subplots(figsize=(13, 5))
for label, s in count_results:
    x = np.linspace(0, HANDS, len(s.earnings_history))
    ax.plot(x, s.earnings_history, lw=1.2, label=label)
ax.axhline(0, color='gray', lw=0.8, linestyle='--')
ax.set_title(f'Earnings by counting system — Basic strategy, {HANDS:,} hands')
ax.set_xlabel('Hands played')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

## 4 · Monte Carlo: distribution of outcomes

In [ ]:
RUNS  = 200
HANDS = 1000
BET   = 25

outcomes_basic = []
outcomes_count = []

for _ in range(RUNS):
    s1 = run_simulation(HANDS, BET, strategy=Strategy.BASIC, counting_system=CountingSystem.NONE)
    s2 = run_simulation(HANDS, BET, strategy=Strategy.BASIC, counting_system=CountingSystem.HILOW, bet_spread=4)
    outcomes_basic.append(s1.net_earnings)
    outcomes_count.append(s2.net_earnings)

fig, ax = plt.subplots(figsize=(11, 4))
ax.hist(outcomes_basic, bins=30, alpha=0.6, label='Basic (no count)', color='#3b82f6')
ax.hist(outcomes_count, bins=30, alpha=0.6, label='Basic + Hi-Lo', color='#22c55e')
ax.axvline(np.mean(outcomes_basic), color='#3b82f6', lw=2, linestyle='--',
           label=f'Basic mean: ${np.mean(outcomes_basic):+,.0f}')
ax.axvline(np.mean(outcomes_count), color='#22c55e', lw=2, linestyle='--',
           label=f'Hi-Lo mean: ${np.mean(outcomes_count):+,.0f}')
ax.set_xlabel(f'Net earnings after {HANDS:,} hands (${BET} bet)')
ax.set_ylabel('Frequency (out of 200 runs)')
ax.set_title(f'Distribution of outcomes — {RUNS} Monte Carlo runs')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Basic    — mean: ${np.mean(outcomes_basic):+,.0f}  std: ${np.std(outcomes_basic):,.0f}  profitable: {sum(v>0 for v in outcomes_basic)}/{RUNS}")
print(f"Hi-Lo    — mean: ${np.mean(outcomes_count):+,.0f}  std: ${np.std(outcomes_count):,.0f}  profitable: {sum(v>0 for v in outcomes_count)}/{RUNS}")

## 5 · Bet spread impact on expected value

In [ ]:
spreads = [1, 2, 4, 6, 8, 12, 16]
edges   = []

for spread in spreads:
    s = run_simulation(
        num_hands=30_000, base_bet=25,
        strategy=Strategy.BASIC,
        counting_system=CountingSystem.HILOW,
        bet_spread=spread,
    )
    edges.append(s.house_edge)
    print(f'Spread 1-{spread:<3}  edge={s.house_edge:+.3f}%  net={s.net_earnings:+,.0f}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(spreads, edges, marker='o', color='#8b5cf6', lw=2)
ax.axhline(0, color='gray', lw=0.8, linestyle='--')
ax.set_xlabel('Bet spread (1 to N)')
ax.set_ylabel('House edge (%)')
ax.set_title('Hi-Lo: house edge vs bet spread — 30,000 hands')
ax.set_xticks(spreads)
for x, y in zip(spreads, edges):
    ax.annotate(f'{y:.2f}%', (x, y), textcoords='offset points', xytext=(0,8), ha='center', fontsize=9)
plt.tight_layout()
plt.show()